## News police Rag system

In this notebook I'll show my workflow into integrating the news we have collected into a RAG system.

### 1- packages

In [ ]:
#%pip install langchain_community langchain_hub chromadb langchain langchain_google_genai pandas sentence-transformers python-dotenv

### 2- APIs

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()  # This loads the .env file
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

### 3- Imports

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

### 4- Treat the documents

in order to treat the documents, we have to make them langchain documents

In [4]:
df = pd.read_csv("../data/7days_transcripts_2025-11-01.csv")
docs = []
content_col = "content"

for _, row in df.iterrows():
    metadata = row.drop(content_col).to_dict()
    page_content = row[content_col]
    docs.append(Document(page_content=page_content, metadata=metadata))

we then split the documents into small chuncks of 1000 chrecters each, this is done becaus eembedding models have limited context windows

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
splits.__len__() 

253

### 5- Indexing and retreiving

here we'll embed the documents, this is is what we'll be using in our RAG, it should be later saved not locally in the chromadb, but in a spcific VectorDB

In [7]:
vectorstore.delete_collection()

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    persist_directory='./chroma_db'
)

retriever = vectorstore.as_retriever()

/var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/ipykernel_15677/1101354665.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


we you can test the retriever with a sentance and it will respond with the documents closest to the question, note that this is a bottleneck for now and I'll try exploring different embedding models to fit our purposes of the algerian dialect

### retrieve from the db

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

# Load existing vectorstore instead of creating new one
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever()

/var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/ipykernel_17705/963267235.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/var/folders/c2/g_xbtkfn0c1g2fbklbv3pmd80000gn/T/ipykernel_17705/963267235.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [5]:
retriever.invoke( "اشكون سرق متحف اللوفر" , k=1)

[Document(metadata={'date': '2025-10-30T18:25:52Z', 'source': 'https://youtu.be/vHALrYzvs88', 'author': 'Nora Trabelsi', 'category': 'politics', 'title': 'اتفاقية1968:المندبة كبيرة والميت فار 🐭🐀🐹🐁😂', 'label': 0}, page_content='مع صحاب الكابه اللي يد 3ور الكيلو بون هذ سي بو غمور فقط ضحك يعني جات في هذا الوقت نتاع سرقه اللوفر اللي قال زيور الجزائر تريد سرقه حضارتنا انتما سرقته سرقته افريقيا راك تقولي على سرقه حضارتنا الحضاره اللي لميتوها من كل الشعوب ودرتوها في متحف وسجات في الاسبوع التي تحاكم فيه الجزائريه كانت كلونديستين وخطفت طفله يعني شوفوا جريمه مروعه يعني الحقيقه يعني انا هذه كون عطاوه لي بين يدي انا نقطعها ماشي ماشي حتى هما يعني باش واحد يكون واقعي يعني جريمه شنعاء في حق طفله مولا 12 سنه والوالدين الطفله مساكن قالوا لا تسيسوا القضيه يعني نو غريكي با لافير قالت الماغين لو بان وجمعتها ما تدخلوش القضيه تاع بنتي في السياسه ومع ذلك سيست وتكلموا عليها في البرلمان علاش قلت لكم انه ما كانش توقيت احسن من هذا التوقيت باش يمر القرار الغاء الاتفاقيات 1968 المهم ما قلت لكم هذا القرار بيد الرئي

### 6- Generation

from a question The model will retrieve relevant docs and pass them to the context of the llm.

In [ ]:
template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Keep the answer concise.

{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)


llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_retries=2
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
rag_chain.invoke("ماهي اهداف ويتكوف و امريكا بين الجزائر و المغرب")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


'حسب عبد القادر بن قرينة، الهدف ليس فقط قضية الصحراء الغربية، بل هنالك هدف أعمق وهو أن الجزائر تدخل في بعضها. أما عن أهداف أمريكا، فهي تريد إقناع الجزائر والصحراء الغربية بمسألة الحكم الذاتي الموسع.'